# 10 — Confidence Model

**Purpose:** Train and evaluate models that estimate P(QT measurement is trustworthy).

**Input:** `outputs/confidence_features.parquet`

**Output:** `outputs/confidence_predictions.parquet`

**Spec:** `CONFIDENCE_MODEL_SPEC.md`

**Models:** XGBoost, HistGradientBoosting (LightGBM equivalent), Random Forest

**Phase 1 Pseudo-targets:** `lead_agreement_score`, `beat_agreement_score`,
`repeatability_score`, `internal_consistency_score`

**Leakage Prevention:** No `absolute_qt_error_ms`, `expert_disagreement_ms`,
`true_t_end_error_ms`, or `confidence_label`.


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from datetime import datetime

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
TIMESTAMP = datetime.utcnow().isoformat()

conf_feat = pd.read_parquet("../outputs/confidence_features.parquet")
print(f"Features loaded: {conf_feat.shape}")
print(conf_feat[["record_id","dataset_name"]].head(3).to_string(index=False))


Features loaded: (70, 107)
  record_id dataset_name
ptbxl/00001        ptbxl
ptbxl/00002        ptbxl
ptbxl/00003        ptbxl


## Feature Preparation

In [3]:
FORBIDDEN_FEATURES = [
    "absolute_qt_error_ms","expert_disagreement_ms","true_t_end_error_ms",
    "measurement_instability_ms","confidence_label","confidence_probability",
]
TARGET_COLS = [
    "mean_lead_agreement_score","mean_beat_agreement_score",
    "mean_repeatability_score","mean_internal_consistency_score",
]

# Check no leakage
leakage = [c for c in FORBIDDEN_FEATURES if c in conf_feat.columns]
assert not leakage, f"Leakage: {leakage}"

# Build pseudo-target: mean of available agreement columns
available_targets = [c for c in TARGET_COLS if c in conf_feat.columns]
print(f"Available target components: {available_targets}")

pseudo_target = conf_feat[available_targets].mean(axis=1)
pseudo_target = pseudo_target.fillna(pseudo_target.median())
# Binarise at median for Phase 1 classification
threshold = float(pseudo_target.median())
y = (pseudo_target >= threshold).astype(int)
print(f"Target distribution: {y.value_counts().to_dict()} (threshold={threshold:.3f})")


Available target components: ['mean_repeatability_score', 'mean_internal_consistency_score']
Target distribution: {1: 35, 0: 35} (threshold=0.409)


In [4]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
cat_cols = ["dataset_name","diagnostic_class","dataset_origin"]
cat_cols_present = [c for c in cat_cols if c in conf_feat.columns]

df_model = conf_feat.copy()
le_dict = {}
for col in cat_cols_present:
    le = LabelEncoder()
    df_model[col] = df_model[col].fillna("unknown")
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

# Drop non-feature cols
DROP_COLS = ["record_id","pipeline_version","processing_timestamp"] + available_targets
feature_cols = [c for c in df_model.columns if c not in DROP_COLS and c not in FORBIDDEN_FEATURES]
X = df_model[feature_cols].fillna(-1).values
print(f"Feature matrix: {X.shape}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:8]} ...")


Feature matrix: (70, 102)
Feature columns (102): ['dataset_name', 'mean_bw_index', 'std_bw_index', 'max_bw_index', 'p95_bw_index', 'mean_hfn_index', 'std_hfn_index', 'max_hfn_index'] ...


## 5-Fold Cross-Validation

In [5]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

MODELS = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        random_state=RANDOM_SEED, eval_metric="logloss", verbosity=0),
    "HistGradBoosting": HistGradientBoostingClassifier(
        max_iter=200, max_depth=4, learning_rate=0.05, random_state=RANDOM_SEED),
}

cv_results = {}
for name, model in MODELS.items():
    oof_proba = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    roc  = roc_auc_score(y, oof_proba)
    prc  = average_precision_score(y, oof_proba)
    bs   = brier_score_loss(y, oof_proba)
    cv_results[name] = {"ROC-AUC": roc, "PR-AUC": prc, "Brier": bs}
    print(f"  {name:22s}  ROC-AUC={roc:.3f}  PR-AUC={prc:.3f}  Brier={bs:.3f}")

results_df = pd.DataFrame(cv_results).T
print("\n5-Fold CV Summary:")
print(results_df.round(3).to_string())


  RandomForest            ROC-AUC=0.988  PR-AUC=0.990  Brier=0.050


  XGBoost                 ROC-AUC=0.978  PR-AUC=0.982  Brier=0.059


  HistGradBoosting        ROC-AUC=0.971  PR-AUC=0.973  Brier=0.066

5-Fold CV Summary:
                  ROC-AUC  PR-AUC  Brier
RandomForest        0.988   0.990  0.050
XGBoost             0.978   0.982  0.059
HistGradBoosting    0.971   0.973  0.066


## Leave-One-Dataset-Out Validation

In [6]:
lodo_results = []
datasets = df_model["dataset_name"].unique() if "dataset_name" in df_model.columns else []

best_model_name = results_df["ROC-AUC"].idxmax()
best_model = MODELS[best_model_name]

print(f"Best model: {best_model_name}")

for ds_code in datasets:
    test_mask  = (df_model["dataset_name"] == ds_code).values
    train_mask = ~test_mask
    if train_mask.sum() < 10 or test_mask.sum() < 5:
        continue
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y.values[train_mask], y.values[test_mask]
    if len(np.unique(y_test)) < 2:
        continue
    best_model.fit(X_train, y_train)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, y_prob)
    ds_name = le_dict["dataset_name"].inverse_transform([ds_code])[0] if "dataset_name" in le_dict else str(ds_code)
    lodo_results.append({"test_dataset": ds_name, "ROC-AUC": roc, "n_test": int(test_mask.sum())})
    print(f"  LODO (test={ds_name:8s})  ROC-AUC={roc:.3f}  n={test_mask.sum()}")

lodo_df = pd.DataFrame(lodo_results)


Best model: RandomForest


  LODO (test=ptbxl   )  ROC-AUC=1.000  n=40


  LODO (test=ludb    )  ROC-AUC=1.000  n=20


  LODO (test=nstdb   )  ROC-AUC=0.875  n=10


## Model Calibration

In [7]:
from sklearn.calibration import CalibrationDisplay

best_model.fit(X, y)
calibrated_model = CalibratedClassifierCV(best_model, method="isotonic", cv=5)
calibrated_model.fit(X, y)
conf_proba_cal = calibrated_model.predict_proba(X)[:, 1]
conf_proba_raw = cross_val_predict(best_model, X, y, cv=5, method="predict_proba")[:, 1]

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
CalibrationDisplay.from_predictions(y, conf_proba_raw, n_bins=10, ax=axes[0], name="Uncalibrated")
CalibrationDisplay.from_predictions(y, conf_proba_cal, n_bins=10, ax=axes[1], name="Isotonic")
axes[0].set_title("Calibration — Uncalibrated")
axes[1].set_title("Calibration — Isotonic Regression")
plt.tight_layout()
plt.savefig("../outputs/calibration_curve.png", dpi=100)
plt.show()
print("Calibration figure saved.")


Calibration figure saved.


## ROC and PR Curves

In [8]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _  = roc_curve(y, conf_proba_cal)
prec, rec, _ = precision_recall_curve(y, conf_proba_cal)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color="#1f77b4", lw=2, label=f"ROC-AUC={roc_auc_score(y, conf_proba_cal):.3f}")
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC Curve (Calibrated)")
axes[0].legend()

axes[1].plot(rec, prec, color="#ff7f0e", lw=2, label=f"PR-AUC={average_precision_score(y, conf_proba_cal):.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="PR Curve (Calibrated)")
axes[1].legend()
plt.tight_layout()
plt.savefig("../outputs/roc_pr_curves.png", dpi=100)
plt.show()
print("ROC/PR figure saved.")


ROC/PR figure saved.


## SHAP Explainability (Top 20 Features)

In [9]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X)

if isinstance(shap_values.values, np.ndarray) and shap_values.values.ndim == 3:
    sv = shap_values.values[:, :, 1]
else:
    sv = shap_values.values

mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
top20 = shap_importance.head(20)

fig, ax = plt.subplots(figsize=(8, 8))
top20[::-1].plot.barh(ax=ax, color="#9467bd", edgecolor="white")
ax.set_title("SHAP Feature Importance (Top 20)")
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig("../outputs/shap_importance.png", dpi=100)
plt.show()
print("SHAP figure saved.")
print("\nTop-10 features:")
print(top20.head(10).round(4).to_string())


SHAP figure saved.

Top-10 features:
mean_bsqi                   0.0619
std_repeatability_score     0.0349
mean_lead_agreement         0.0347
mean_qt_variance_leads      0.0317
max_repeatability_score     0.0315
p95_repeatability_score     0.0261
mean_qt_variance_beats      0.0261
p95_beat_agreement_score    0.0233
mean_beat_agreement         0.0225
max_bsqi                    0.0210


## Export Predictions

In [10]:
predictions = conf_feat[["record_id","dataset_name"]].copy()
predictions["confidence_probability"] = conf_proba_cal
predictions["model_name"]        = best_model_name
predictions["calibration_method"] = "isotonic"
predictions["pipeline_version"]   = PIPELINE_VERSION
predictions["processing_timestamp"] = TIMESTAMP

REQUIRED_PRED_COLS = ["record_id","confidence_probability","model_name","calibration_method","pipeline_version"]
missing = [c for c in REQUIRED_PRED_COLS if c not in predictions.columns]
assert not missing, f"Missing: {missing}"
assert predictions["confidence_probability"].between(0,1).all(), "Probability out of [0,1]"

predictions.to_parquet("../outputs/confidence_predictions.parquet", index=False)
print("✓ confidence_predictions.parquet →", predictions.shape)
print(predictions["confidence_probability"].describe().round(3).to_string())


✓ confidence_predictions.parquet → (70, 7)
count    70.000
mean      0.494
std       0.495
min       0.000
25%       0.000
50%       0.457
75%       1.000
max       1.000


## Performance Summary

In [11]:
print("=" * 55)
print("CONFIDENCE MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print("\n5-Fold CV Results:")
print(results_df.round(3).to_string())
if not lodo_df.empty:
    print("\nLeave-One-Dataset-Out:")
    print(lodo_df.round(3).to_string(index=False))
print(f"\nCalibrated model: {best_model_name} + Isotonic Regression")
print(f"Pipeline version: {PIPELINE_VERSION}")


CONFIDENCE MODEL PERFORMANCE SUMMARY

5-Fold CV Results:
                  ROC-AUC  PR-AUC  Brier
RandomForest        0.988   0.990  0.050
XGBoost             0.978   0.982  0.059
HistGradBoosting    0.971   0.973  0.066

Leave-One-Dataset-Out:
test_dataset  ROC-AUC  n_test
       ptbxl    1.000      40
        ludb    1.000      20
       nstdb    0.875      10

Calibrated model: RandomForest + Isotonic Regression
Pipeline version: 1.0.0
